# PEPS与CTMRG算法

本教程介绍二维张量网络PEPS和角转移矩阵重整化群(CTMRG)算法。

## 学习目标

1. 理解PEPS张量网络结构
2. 学习CTMRG算法原理
3. 实现简单更新(Simple Update)
4. 计算物理量期望值

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle
from matplotlib.lines import Line2D
import sys

sys.path.append('../../common')
from utils.tensor_utils import entanglement_entropy

plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=4, suppress=True)

## 1. PEPS张量网络

### 为什么需要PEPS？

- **MPS**: 适用于1D系统，面积律 S ~ 1
- **PEPS**: 适用于2D系统，面积律 S ~ L

### PEPS张量结构

每个格点一个张量 $T^{i,j}_{u,d,l,r,s}$：

```
      u
      |
  l---●---r    ● = PEPS tensor
      |        s = physical index (d)
      d        u,d,l,r = virtual indices (D)
```

形状：$(D, D, D, D, d)$

In [ ]:
def visualize_peps_network(Lx=4, Ly=4):
    """可视化PEPS网络结构"""
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 绘制格点和键
    for i in range(Lx):
        for j in range(Ly):
            x, y = i, j
            
            # 绘制PEPS张量（圆圈）
            circle = Circle((x, y), 0.15, color='lightblue', 
                          ec='black', linewidth=2, zorder=3)
            ax.add_patch(circle)
            
            # 物理指标（向下）
            ax.plot([x, x], [y-0.15, y-0.4], 'k-', linewidth=2)
            ax.plot(x, y-0.4, 'ro', markersize=8, zorder=4)
            
            # 虚指标（水平和竖直）
            if i < Lx - 1:
                ax.plot([x+0.15, x+0.85], [y, y], 'b-', linewidth=2.5)
            if j < Ly - 1:
                ax.plot([x, x], [y+0.15, y+0.85], 'b-', linewidth=2.5)
    
    # 标注
    ax.text(0, -0.7, 'Physical\nIndex', ha='center', fontsize=10, color='red')
    ax.text(0.5, 0.5, 'Virtual\nBond', ha='center', fontsize=10, color='blue',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    
    ax.set_xlim(-0.5, Lx-0.5)
    ax.set_ylim(-1, Ly-0.5)
    ax.set_aspect('equal')
    ax.set_title(f'PEPS Network Structure ({Lx}×{Ly})\n' + 
                'Blue: Virtual bonds (D), Red: Physical indices (d)',
                fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')
    
    plt.tight_layout()
    return fig

visualize_peps_network()
plt.show()

## 2. PEPS收缩问题

### 为什么困难？

计算 $\langle \psi | \psi \rangle$ 需要收缩2D网络：

**复杂度**：$O(D^{2L})$ - 指数难！

### 解决方案

1. **简单更新(SU)**: 独立更新张量，忽略环境
2. **CTMRG**: 使用环境张量近似
3. **全更新(FU)**: 精确考虑环境（更昂贵）

## 3. CTMRG算法

### 核心思想

用有限大小的环境张量近似无限边界：

```
C----T----C
|    |    |
T----●----T    C = Corner tensor (χ×χ)
|    |    |    T = Transfer tensor (χ×D×χ)
C----T----C    ● = PEPS tensor (D×D×D×D×d)
```

χ: 环境键维度（截断）

In [ ]:
def visualize_ctmrg_environment():
    """可视化CTMRG环境张量"""
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 中心PEPS张量
    center = Circle((0, 0), 0.3, color='lightblue', 
                   ec='black', linewidth=3, zorder=4)
    ax.add_patch(center)
    ax.text(0, 0, '●\nPEPS', ha='center', va='center', 
           fontsize=12, fontweight='bold')
    
    # 角张量 (Corners)
    corner_positions = [(-2, 2), (2, 2), (2, -2), (-2, -2)]
    corner_labels = ['C₁', 'C₂', 'C₃', 'C₄']
    
    for pos, label in zip(corner_positions, corner_labels):
        rect = FancyBboxPatch((pos[0]-0.4, pos[1]-0.4), 0.8, 0.8,
                             boxstyle="round,pad=0.1",
                             facecolor='lightcoral', 
                             edgecolor='black', linewidth=2, zorder=3)
        ax.add_patch(rect)
        ax.text(pos[0], pos[1], label, ha='center', va='center',
               fontsize=12, fontweight='bold')
    
    # 边张量 (Transfers)
    transfer_positions = [(0, 2), (2, 0), (0, -2), (-2, 0)]
    transfer_labels = ['T₁', 'T₂', 'T₃', 'T₄']
    
    for pos, label in zip(transfer_positions, transfer_labels):
        rect = FancyBboxPatch((pos[0]-0.4, pos[1]-0.3), 0.8, 0.6,
                             boxstyle="round,pad=0.05",
                             facecolor='lightgreen',
                             edgecolor='black', linewidth=2, zorder=3)
        ax.add_patch(rect)
        ax.text(pos[0], pos[1], label, ha='center', va='center',
               fontsize=12, fontweight='bold')
    
    # 连接线
    # 上边
    ax.plot([-1.6, -0.4], [2, 2], 'k-', linewidth=2)
    ax.plot([0.4, 1.6], [2, 2], 'k-', linewidth=2)
    ax.plot([0, 0], [1.7, 0.3], 'k-', linewidth=2)
    
    # 右边
    ax.plot([2, 2], [1.6, 0.3], 'k-', linewidth=2)
    ax.plot([2, 2], [-0.3, -1.6], 'k-', linewidth=2)
    ax.plot([1.7, 0.3], [0, 0], 'k-', linewidth=2)
    
    # 下边
    ax.plot([1.6, 0.4], [-2, -2], 'k-', linewidth=2)
    ax.plot([-0.4, -1.6], [-2, -2], 'k-', linewidth=2)
    ax.plot([0, 0], [-1.7, -0.3], 'k-', linewidth=2)
    
    # 左边
    ax.plot([-2, -2], [-1.6, -0.3], 'k-', linewidth=2)
    ax.plot([-2, -2], [0.3, 1.6], 'k-', linewidth=2)
    ax.plot([-1.7, -0.3], [0, 0], 'k-', linewidth=2)
    
    # 图例
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='lightblue', markersize=15,
              label='PEPS Tensor'),
        Line2D([0], [0], marker='s', color='w',
              markerfacecolor='lightcoral', markersize=15,
              label='Corner (C): χ×χ'),
        Line2D([0], [0], marker='s', color='w',
              markerfacecolor='lightgreen', markersize=15,
              label='Transfer (T): χ×D×χ')
    ]
    ax.legend(handles=legend_elements, loc='upper left', 
             fontsize=11, framealpha=0.9)
    
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title('CTMRG Environment Tensors\n' +
                '(Approximating infinite boundary with finite χ)',
                fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')
    
    plt.tight_layout()
    return fig

visualize_ctmrg_environment()
plt.show()

## 4. CTMRG算法步骤

### 迭代过程

1. **初始化**: 随机C和T（或从简单形式开始）
2. **左移(Left Move)**:
   - 在左边插入一列PEPS张量
   - 收缩并截断到χ维度
3. **上移(Up Move)**: 类似
4. **右移(Right Move)**: 类似
5. **下移(Down Move)**: 类似
6. **重复**: 直到收敛

### 收缩示意

In [ ]:
def demonstrate_ctmrg_move():
    """演示CTMRG移动步骤"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    steps = ['Before', 'Insert Column', 'After Truncation']
    
    for idx, (ax, step) in enumerate(zip(axes, steps)):
        ax.set_xlim(0, 4)
        ax.set_ylim(0, 3)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(step, fontsize=13, fontweight='bold')
        
        if idx == 0:
            # Before: original environment
            # Top corner
            ax.add_patch(Rectangle((0.5, 2), 0.5, 0.5, 
                                  facecolor='lightcoral', ec='black', lw=2))
            ax.text(0.75, 2.25, 'C', ha='center', va='center', fontweight='bold')
            
            # Bottom corner
            ax.add_patch(Rectangle((0.5, 0.5), 0.5, 0.5,
                                  facecolor='lightcoral', ec='black', lw=2))
            ax.text(0.75, 0.75, 'C', ha='center', va='center', fontweight='bold')
            
            # Transfer tensors
            for y in [1.2, 1.8]:
                ax.add_patch(Rectangle((0.5, y), 0.5, 0.3,
                                      facecolor='lightgreen', ec='black', lw=2))
                ax.text(0.75, y+0.15, 'T', ha='center', va='center', fontweight='bold')
            
        elif idx == 1:
            # Insert: add new column
            # Old environment
            ax.add_patch(Rectangle((0.2, 2), 0.3, 0.5,
                                  facecolor='lightcoral', ec='black', lw=1.5, alpha=0.5))
            ax.add_patch(Rectangle((0.2, 0.5), 0.3, 0.5,
                                  facecolor='lightcoral', ec='black', lw=1.5, alpha=0.5))
            
            # New PEPS column
            for y in [2.25, 1.5, 0.75]:
                circle = Circle((1.5, y), 0.2, color='lightblue',
                              ec='black', linewidth=2)
                ax.add_patch(circle)
                ax.text(1.5, y, '●', ha='center', va='center', fontsize=14)
            
            # Arrows showing contraction
            ax.annotate('', xy=(1.0, 1.5), xytext=(0.6, 1.5),
                       arrowprops=dict(arrowstyle='->', lw=2, color='red'))
            
        else:
            # After: updated environment
            ax.add_patch(Rectangle((1.5, 2), 0.5, 0.5,
                                  facecolor='orange', ec='black', lw=2))
            ax.text(1.75, 2.25, "C'", ha='center', va='center', fontweight='bold')
            
            ax.add_patch(Rectangle((1.5, 0.5), 0.5, 0.5,
                                  facecolor='orange', ec='black', lw=2))
            ax.text(1.75, 0.75, "C'", ha='center', va='center', fontweight='bold')
            
            for y in [1.2, 1.8]:
                ax.add_patch(Rectangle((1.5, y), 0.5, 0.3,
                                      facecolor='yellow', ec='black', lw=2))
                ax.text(1.75, y+0.15, "T'", ha='center', va='center', fontweight='bold')
            
            ax.text(2.5, 1.5, 'Truncated\nto χ', ha='center', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='wheat'))
    
    plt.suptitle('CTMRG Left Move', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

demonstrate_ctmrg_move()
plt.show()

## 5. 简单实现

### 初始化PEPS张量

In [ ]:
def initialize_peps_tensor(D, d, random_state=42):
    """
    初始化随机PEPS张量
    
    参数:
        D: 虚指标维度
        d: 物理指标维度
        random_state: 随机种子
    
    返回:
        T: PEPS张量 (D, D, D, D, d)
    """
    np.random.seed(random_state)
    T = np.random.randn(D, D, D, D, d) + 1j * np.random.randn(D, D, D, D, d)
    
    # 归一化
    T = T / np.linalg.norm(T)
    
    return T

def initialize_corner(chi, random_state=42):
    """
    初始化角张量
    
    参数:
        chi: 环境键维度
    
    返回:
        C: 角张量 (chi, chi)
    """
    np.random.seed(random_state)
    C = np.eye(chi, dtype=complex)  # 从单位矩阵开始
    return C

def initialize_transfer(chi, D, random_state=42):
    """
    初始化边张量
    
    参数:
        chi: 环境键维度
        D: PEPS虚指标维度
    
    返回:
        T: 边张量 (chi, D, chi)
    """
    np.random.seed(random_state)
    T = np.random.randn(chi, D, chi) + 1j * np.random.randn(chi, D, chi)
    T = T / np.linalg.norm(T)
    return T

# 示例
D = 3   # PEPS键维度
d = 2   # 物理维度（自旋-1/2）
chi = 20  # 环境键维度

peps_tensor = initialize_peps_tensor(D, d)
corner = initialize_corner(chi)
transfer = initialize_transfer(chi, D)

print("PEPS张量初始化")
print("="*50)
print(f"PEPS张量形状: {peps_tensor.shape}")
print(f"角张量形状: {corner.shape}")
print(f"边张量形状: {transfer.shape}")
print(f"\nPEPS张量范数: {np.linalg.norm(peps_tensor):.4f}")
print(f"物理指标维度: d = {d}")
print(f"虚指标维度: D = {D}")
print(f"环境键维度: χ = {chi}")

## 6. 计算期望值

### 单点期望值

计算 $\langle O \rangle = \frac{\langle \psi | O | \psi \rangle}{\langle \psi | \psi \rangle}$

使用CTMRG环境：

In [ ]:
def compute_local_expectation(peps_tensor, operator, corners, transfers):
    """
    计算局域算符期望值（简化版）
    
    参数:
        peps_tensor: PEPS张量
        operator: 物理算符 (d, d)
        corners: 四个角张量列表
        transfers: 四个边张量列表
    
    返回:
        期望值
    """
    # 这里是简化实现
    # 完整实现需要收缩整个环境
    
    d = peps_tensor.shape[-1]
    
    # 简化：只看PEPS张量的约化密度矩阵
    # 完整版需要包含环境收缩
    
    # 收缩虚指标（简化）
    rho = np.einsum('ijkls,ijklt->st', peps_tensor, peps_tensor.conj())
    rho = rho / np.trace(rho)
    
    # 期望值
    expectation = np.trace(operator @ rho)
    
    return expectation.real

# 示例：计算Sz期望值
Sz = np.array([[0.5, 0], [0, -0.5]])

# 需要角和边张量（这里用占位符）
corners_list = [corner for _ in range(4)]
transfers_list = [transfer for _ in range(4)]

exp_Sz = compute_local_expectation(peps_tensor, Sz, corners_list, transfers_list)

print("\n期望值计算")
print("="*50)
print(f"⟨Sz⟩ = {exp_Sz:.6f}")
print("\n注意：这是简化版本")
print("完整CTMRG需要：")
print("1. 迭代更新环境张量")
print("2. 收缩完整环境网络")
print("3. 归一化和截断")

## 7. CTMRG收敛性

### 监控指标

1. **能量收敛**: $|E_n - E_{n-1}| < \epsilon$
2. **环境收敛**: $||C_n - C_{n-1}|| < \delta$
3. **奇异值谱**: 检查截断误差

In [ ]:
def simulate_ctmrg_convergence(n_iterations=50):
    """
    模拟CTMRG收敛过程
    """
    # 模拟能量收敛
    energies = []
    E_exact = -1.5  # 假设的精确能量
    
    for i in range(n_iterations):
        # 指数收敛
        E = E_exact + 0.5 * np.exp(-i/10) + 0.01 * np.random.randn()
        energies.append(E)
    
    energies = np.array(energies)
    
    # 绘图
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 能量收敛
    iterations = np.arange(n_iterations)
    ax1.plot(iterations, energies, 'o-', markersize=4, linewidth=2, label='CTMRG')
    ax1.axhline(E_exact, color='red', linestyle='--', linewidth=2, label='Exact')
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Energy per site', fontsize=12)
    ax1.set_title('CTMRG Energy Convergence', fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 误差（对数）
    errors = np.abs(energies - E_exact)
    ax2.semilogy(iterations, errors, 'o-', markersize=4, linewidth=2, color='green')
    ax2.axhline(1e-6, color='red', linestyle='--', linewidth=2, label='Threshold')
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('|E - E_exact|', fontsize=12)
    ax2.set_title('Convergence Error (log scale)', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # 统计
    print("\nCTMRG收敛分析")
    print("="*50)
    print(f"初始能量: {energies[0]:.6f}")
    print(f"最终能量: {energies[-1]:.6f}")
    print(f"精确能量: {E_exact:.6f}")
    print(f"最终误差: {errors[-1]:.2e}")
    
    # 收敛迭代次数
    converged_idx = np.where(errors < 1e-4)[0]
    if len(converged_idx) > 0:
        print(f"收敛迭代数 (ε=1e-4): {converged_idx[0]}")

simulate_ctmrg_convergence()

## 8. χ依赖性

### 环境键维度的影响

- **小χ**: 快速但不准确
- **大χ**: 准确但昂贵
- **权衡**: χ ~ 10D 通常足够

In [ ]:
def plot_chi_dependence():
    """绘制χ依赖性"""
    chi_values = np.array([10, 20, 30, 40, 50, 60, 80, 100])
    
    # 模拟数据
    E_exact = -1.5
    energies = E_exact + 0.5 / chi_values + 0.01 * np.random.randn(len(chi_values))
    times = (chi_values / 10) ** 3  # 立方复杂度
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 精度 vs χ
    ax1.plot(chi_values, energies, 'o-', markersize=8, linewidth=2)
    ax1.axhline(E_exact, color='red', linestyle='--', linewidth=2, label='Exact')
    ax1.set_xlabel('Environment bond dimension χ', fontsize=12)
    ax1.set_ylabel('Energy per site', fontsize=12)
    ax1.set_title('Accuracy vs χ', fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 时间 vs χ
    ax2.loglog(chi_values, times, 's-', markersize=8, linewidth=2, color='green')
    # 拟合线
    ax2.loglog(chi_values, 0.01 * chi_values**3, '--', linewidth=2, 
              color='red', label='O(χ³)')
    ax2.set_xlabel('Environment bond dimension χ', fontsize=12)
    ax2.set_ylabel('Time per iteration (a.u.)', fontsize=12)
    ax2.set_title('Computational Cost vs χ', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print("\n环境键维度选择指南")
    print("="*50)
    print("χ = 10D  : 快速探索")
    print("χ = 20D  : 标准精度")
    print("χ = 50D  : 高精度")
    print("χ > 100D : 基准计算")
    print("\n复杂度: O(χ³ D³)")

plot_chi_dependence()

## 总结

### PEPS要点

1. **2D推广**: MPS → PEPS
2. **面积律**: 2D纠缠熵 S ~ L
3. **收缩困难**: 指数复杂度

### CTMRG要点

1. **环境近似**: C和T张量
2. **迭代更新**: 四个方向移动
3. **截断**: 控制χ
4. **收敛**: 监控能量和环境

### 实际应用

- Kitaev蜂窝模型
- 2D量子自旋系统
- 拓扑序识别
- 关联函数计算

## 下一步

1. 完整CTMRG实现
2. 计算拓扑纠缠熵
3. 识别任意子
4. 相图研究

## 练习

1. 实现完整的CTMRG左移步骤
2. 计算不同χ的能量
3. 监控截断误差
4. 优化收缩顺序

## 参考文献

1. Verstraete, F. & Cirac, J. I. (2004). *Renormalization algorithms for Quantum-Many Body Systems*.
2. Jiang, H. C. et al. (2008). *Accurate Determination of Tensor Network State*. PRL.
3. Nishino, T. & Okunishi, K. (1996). *Corner Transfer Matrix Algorithm*. JPSJ.